# Final Integration Task — Multimodal Incident Report Analyzer

**Assignment:** AI for Engineers — Multimodal Crime / Incident Report Analyzer  
**Task:** Team Final Integration — merge all 5 modality outputs into one unified incident dataset

## Integration Steps (from assignment)
1. Define a common `Incident_ID` key across all five output CSVs
2. Merge all five DataFrames using pandas `merge/join` on `Incident_ID`
3. Handle missing values where a modality has no data for a given incident
4. Generate a final severity classification (Low / Medium / High) based on combined signals
5. Build a simple dashboard or query interface to display and filter incident summaries

## Final Integrated Dataset Structure
| Incident_ID | Audio_Event | PDF_Doc_Type | Image_Objects | Video_Event | Text_Crime_Type | Severity |
|-------------|-------------|--------------|---------------|-------------|-----------------|----------|
| INC_001 | Fire, Medical Emergency | 1033 MRAP Training Proposal | No data | Person browsing | Fire / Arson | High |

In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
!pip install pandas tabulate -q
print('Ready.')

In [ ]:
# ============================================================
# CELL 2 — Load all five modality output CSVs
# ============================================================
import pandas as pd
import os

# Adjust paths if running from a different working directory
BASE = os.path.join(os.path.dirname(os.getcwd()), '') if 'integration' not in os.getcwd() else '..'

audio_df = pd.read_csv(os.path.join(BASE, 'audio',  'audio_output.csv'))
pdf_df   = pd.read_csv(os.path.join(BASE, 'pdf',    'pdf_output.csv'))
image_df = pd.read_csv(os.path.join(BASE, 'images', 'image_output.csv'))
video_df = pd.read_csv(os.path.join(BASE, 'video',  'video_output.csv'))
text_df  = pd.read_csv(os.path.join(BASE, 'text',   'text_output.csv'))

print(f'Audio:  {len(audio_df)} rows | columns: {list(audio_df.columns)}')
print(f'PDF:    {len(pdf_df)} rows | columns: {list(pdf_df.columns)}')
print(f'Image:  {len(image_df)} rows | columns: {list(image_df.columns)}')
print(f'Video:  {len(video_df)} rows | columns: {list(video_df.columns)}')
print(f'Text:   {len(text_df)} rows | columns: {list(text_df.columns)}')

In [ ]:
# ============================================================
# STEP 1 — Define a common Incident_ID key across all five CSVs
# 5 incidents anchored on audio call types; each modality
# source is mapped to the most semantically matching incident.
# ============================================================

# Audio call IDs → Incident_ID (1:1, audio is primary source)
AUDIO_MAP = {
    'C001': 'INC_001',   # Fire + Medical Emergency
    'C002': 'INC_002',   # Road Accident + Medical Emergency
    'C003': 'INC_003',   # Robbery / Theft
    'C004': 'INC_004',   # Assault / Fight
    'C005': 'INC_005',   # Fire / Building fire
}

# PDF report IDs → Incident_ID (matched by index / department)
PDF_MAP = {
    'RPT_001': 'INC_001',
    'RPT_002': 'INC_002',
    'RPT_003': 'INC_003',
    'RPT_004': 'INC_004',
    'RPT_005': 'INC_005',
}

# Image IDs → Incident_ID (matched by scene type)
IMAGE_MAP = {
    'IMG_001': 'INC_002',   # Road Incident Scene  → Road Accident
    'IMG_002': 'INC_004',   # Public Disturbance   → Assault / Fight
}

# Video clip IDs → Incident_ID (matched by dominant detected event)
VIDEO_MAP = {
    'CAVIAR_01': 'INC_003',  # Suspicious movement  → Robbery
    'CAVIAR_02': 'INC_004',  # Suspicious movement  → Assault
    'CAVIAR_03': 'INC_001',  # Person in scene      → Fire/Emergency
}

# Text post IDs → Incident_ID (matched by crime type to incident)
TEXT_MAP = {
    'INC_001': 30,    # TXT_031 Fire / Arson
    'INC_002': 0,     # TXT_001 Active crime / road scene
    'INC_003': 6,     # TXT_007 Robbery / Theft
    'INC_004': 12,    # TXT_013 Assault
    'INC_005': 104,   # TXT_105 Fire / Arson
}

print('Incident_ID mappings defined for all 5 modalities.')
print('Incidents: INC_001 (Fire/Medical) | INC_002 (Road Accident) | '
      'INC_003 (Robbery) | INC_004 (Assault) | INC_005 (Building Fire)')

In [ ]:
# ============================================================
# STEP 1 (cont.) — Tag each modality DataFrame with Incident_ID
# ============================================================

# --- Audio ---
audio_inc = audio_df.copy()
audio_inc['Incident_ID'] = audio_inc['Call_ID'].map(AUDIO_MAP)
audio_inc = audio_inc[['Incident_ID','Call_ID','Extracted_Event','Location','Urgency_Score']]
audio_inc.columns = ['Incident_ID','Audio_Source','Audio_Event','Audio_Location','Audio_Urgency']

# --- PDF ---
pdf_inc = pdf_df.copy()
pdf_inc['Incident_ID'] = pdf_inc['Report_ID'].map(PDF_MAP)
pdf_inc = pdf_inc[['Incident_ID','Report_ID','Department','Doc_Type','Date','Program']]
pdf_inc.columns = ['Incident_ID','PDF_Source','PDF_Department','PDF_Doc_Type','PDF_Date','PDF_Program']

# --- Image ---
image_inc = image_df.copy()
image_inc['Incident_ID'] = image_inc['Image_ID'].map(IMAGE_MAP)
image_inc = image_inc[['Incident_ID','Image_ID','Scene_Type','Objects_Detected','Confidence']]
image_inc.columns = ['Incident_ID','Image_Source','Image_Scene_Type','Image_Objects','Image_Confidence']

# --- Video: aggregate per clip (dominant event, max persons, avg confidence) ---
video_agg = (
    video_df.groupby('Clip_ID')
    .agg(
        Video_Event       = ('Event_Detected', lambda x: x.mode()[0]),
        Video_Max_Persons = ('Persons_Count',  'max'),
        Video_Avg_Conf    = ('Confidence',      lambda x: round(x.mean(), 2)),
        Video_Frame_Count = ('Frame_ID',        'count'),
    )
    .reset_index()
)
video_agg['Incident_ID'] = video_agg['Clip_ID'].map(VIDEO_MAP)
video_agg.rename(columns={'Clip_ID': 'Video_Source'}, inplace=True)

# --- Text: pick best-matching record per incident ---
text_rows = []
for inc_id, idx in TEXT_MAP.items():
    row = text_df.iloc[idx].copy()
    row['Incident_ID'] = inc_id
    text_rows.append(row)
text_inc = pd.DataFrame(text_rows)[['Incident_ID','Text_ID','Crime_Type','Location_Entity','Sentiment','Topic','Severity_Label']]
text_inc.columns = ['Incident_ID','Text_Source','Text_Crime_Type','Text_Location','Text_Sentiment','Text_Topic','Text_Severity']
text_inc = text_inc.reset_index(drop=True)

print('All 5 DataFrames tagged with Incident_ID.')

In [ ]:
# ============================================================
# STEP 2 — Merge all five DataFrames using pandas merge/join
# ============================================================

# Base: one row per incident
base_df = pd.DataFrame({'Incident_ID': ['INC_001','INC_002','INC_003','INC_004','INC_005']})

merged = (
    base_df
    .merge(audio_inc,  on='Incident_ID', how='left')
    .merge(pdf_inc,    on='Incident_ID', how='left')
    .merge(image_inc,  on='Incident_ID', how='left')
    .merge(video_agg,  on='Incident_ID', how='left')
    .merge(text_inc,   on='Incident_ID', how='left')
)

print(f'Merged shape: {merged.shape}  ({len(merged)} incidents x {len(merged.columns)} columns)')
print('\nKey columns preview:')
print(merged[['Incident_ID','Audio_Event','PDF_Doc_Type',
              'Image_Objects','Video_Event','Text_Crime_Type']].to_string(index=False))

In [ ]:
# ============================================================
# STEP 3 — Handle missing values
# Where a modality has no data for a given incident,
# fill with 'No data' (string cols) or 0 (numeric cols)
# ============================================================

print('Missing values before fill:')
print(merged.isnull().sum()[merged.isnull().sum() > 0])

# String columns → 'No data'
str_cols = merged.select_dtypes(include='object').columns
merged[str_cols] = merged[str_cols].fillna('No data')

# Numeric columns → 0
num_cols = merged.select_dtypes(include='number').columns
merged[num_cols] = merged[num_cols].fillna(0)

print('\nMissing values after fill:')
print(merged.isnull().sum().sum(), '(should be 0)')

In [ ]:
# ============================================================
# STEP 4 — Final severity classification (Low / Medium / High)
# Based on combined signals from all 5 modalities
# ============================================================

HIGH_KEYWORDS = [
    'fire','medical emergency','assault','fight','homicide',
    'shooting','weapon','trapped','dead','arson','attack'
]
MEDIUM_KEYWORDS = [
    'robbery','theft','accident','crash','suspicious',
    'disturbance','drug'
]

def compute_severity(row):
    """
    Combines signals from all 5 modalities:
    - Audio urgency score (0–1)
    - Text severity label
    - Keywords found in audio event, text crime type, video event, image scene
    """
    signal_text = ' '.join([
        str(row.get('Audio_Event',    '')),
        str(row.get('Text_Crime_Type','')),
        str(row.get('Text_Severity',  '')),
        str(row.get('Image_Scene_Type','')),
        str(row.get('Video_Event',    '')),
    ]).lower()

    urgency = float(row.get('Audio_Urgency', 0)) if str(row.get('Audio_Urgency','')) != 'No data' else 0.0

    if any(w in signal_text for w in HIGH_KEYWORDS) or urgency >= 0.75:
        return 'High'
    if any(w in signal_text for w in MEDIUM_KEYWORDS) or urgency >= 0.4:
        return 'Medium'
    return 'Low'

merged['Severity'] = merged.apply(compute_severity, axis=1)

print('Severity distribution:')
print(merged['Severity'].value_counts())
print()
print(merged[['Incident_ID','Audio_Event','Severity']].to_string(index=False))

In [ ]:
# ============================================================
# Save final_incident_report.csv
# ============================================================

FINAL_COLS = [
    'Incident_ID',
    'Audio_Source',  'Audio_Event',         'Audio_Location', 'Audio_Urgency',
    'PDF_Source',    'PDF_Department',       'PDF_Doc_Type',   'PDF_Date',     'PDF_Program',
    'Image_Source',  'Image_Scene_Type',     'Image_Objects',  'Image_Confidence',
    'Video_Source',  'Video_Event',          'Video_Max_Persons', 'Video_Avg_Conf', 'Video_Frame_Count',
    'Text_Source',   'Text_Crime_Type',      'Text_Location',  'Text_Sentiment', 'Text_Topic', 'Text_Severity',
    'Severity'
]

final_df = merged[FINAL_COLS]

assert list(final_df.columns) == FINAL_COLS, f'Column mismatch!'
assert len(final_df) == 5, f'Expected 5 incidents, got {len(final_df)}'
assert final_df.isnull().sum().sum() == 0, 'Missing values found!'

final_df.to_csv('final_incident_report.csv', index=False)
print(f'Saved final_incident_report.csv  ({len(final_df)} rows x {len(final_df.columns)} columns)')
print()
# Print the summary view matching the assignment's expected structure
summary = final_df[['Incident_ID','Audio_Event','PDF_Doc_Type','Image_Objects','Video_Event','Text_Crime_Type','Severity']]
print(summary.to_string(index=False))

In [ ]:
# ============================================================
# STEP 5 — Query Interface
# Filter and display incident summaries by severity, event type,
# or location. Demonstrates the unified pipeline output.
# ============================================================

def query_incidents(df, severity=None, event_keyword=None, location=None):
    """
    Filter the integrated incident dataset.

    Parameters
    ----------
    severity       : 'High' | 'Medium' | 'Low'  (or None for all)
    event_keyword  : substring to match in Audio_Event (case-insensitive)
    location       : substring to match in Audio_Location (case-insensitive)

    Returns
    -------
    Filtered DataFrame with summary columns
    """
    result = df.copy()

    if severity:
        result = result[result['Severity'] == severity]
    if event_keyword:
        result = result[result['Audio_Event'].str.contains(event_keyword, case=False, na=False)]
    if location:
        result = result[result['Audio_Location'].str.contains(location, case=False, na=False)]

    return result[['Incident_ID','Audio_Event','PDF_Doc_Type',
                   'Image_Objects','Video_Event','Text_Crime_Type','Severity']]

def print_report(label, result):
    print(f'\n{'='*70}')
    print(f' QUERY: {label}  ({len(result)} result(s))')
    print(f'{'='*70}')
    if len(result) == 0:
        print('  No incidents match this query.')
    else:
        print(result.to_string(index=False))

# --- Run sample queries ---
print_report('All HIGH severity incidents',
             query_incidents(final_df, severity='High'))

print_report('All incidents involving FIRE',
             query_incidents(final_df, event_keyword='fire'))

print_report('All incidents on ROAD / HIGHWAY',
             query_incidents(final_df, event_keyword='road accident'))

print_report('All MEDIUM severity incidents',
             query_incidents(final_df, severity='Medium'))

print_report('Full incident summary (all)',
             query_incidents(final_df))

In [ ]:
# ============================================================
# BONUS — Incident statistics dashboard (text-based)
# ============================================================

print('\n' + '='*60)
print('   MULTIMODAL INCIDENT REPORT — SUMMARY DASHBOARD')
print('='*60)
print(f'  Total incidents analysed  : {len(final_df)}')
print(f'  High severity             : {(final_df["Severity"]=="High").sum()}')
print(f'  Medium severity           : {(final_df["Severity"]=="Medium").sum()}')
print(f'  Low severity              : {(final_df["Severity"]=="Low").sum()}')
print()
print('  Incident breakdown:')
for _, row in final_df.iterrows():
    print(f'  [{row["Severity"]:6s}] {row["Incident_ID"]} | '
          f'{row["Audio_Event"][:35]:35s} | '
          f'Urgency={row["Audio_Urgency"]} | '
          f'Text: {row["Text_Crime_Type"]}')
print()
print('  Modality coverage per incident:')
print(f'  {"Incident_ID":12s} {"Audio":6s} {"PDF":6s} {"Image":6s} {"Video":6s} {"Text":6s}')
for _, row in final_df.iterrows():
    audio = 'YES' if row['Audio_Source'] != 'No data' else 'NO '
    pdf   = 'YES' if row['PDF_Source']   != 'No data' else 'NO '
    img   = 'YES' if row['Image_Source'] != 'No data' else 'NO '
    vid   = 'YES' if row['Video_Source'] != 'No data' else 'NO '
    txt   = 'YES' if row['Text_Source']  != 'No data' else 'NO '
    print(f'  {row["Incident_ID"]:12s} {audio:6s} {pdf:6s} {img:6s} {vid:6s} {txt:6s}')
print('='*60)